# Summary_Day7.ipynb  
## 선형회귀 · nn.Linear · 커스텀 모델 · 머신러닝/딥러닝 기본 이론

이번 7차시는 **선형회귀 Linear Regression**를 중심으로, PyTorch의 `nn.Linear`와 커스텀 모델 정의 방식을 정리합니다.

핵심 목표:

1. 회귀와 분류의 차이 이해
2. 선형 함수 `y = Wx + b` 이해
3. `nn.Linear(in_features, out_features)` 구조 이해
4. weight와 bias 확인 및 초기화
5. 1입력 1출력, 2입력 1출력, 2입력 3출력 선형 함수 실습
6. `nn.Module`을 상속한 커스텀 모델 정의
7. `__init__`과 `forward()` 역할 이해
8. MSELoss로 손실 계산
9. 경사하강법 학습 루프 구현
10. 단순 회귀에서 다중 회귀로 확장
11. 학습률이 너무 클 때 발산하는 문제 이해
12. R² Score, 과적합, Ridge/Lasso/ElasticNet 개념 정리
13. Scikit-Learn과 PyTorch의 기본 머신러닝 코드 흐름 비교

핵심 흐름:

```text
데이터 준비 → 모델 정의 → 손실 함수 → Optimizer → 학습 → 평가
```

## 1. 라이브러리 준비

이번 실습에서는 NumPy, Matplotlib, PyTorch, scikit-learn을 사용합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris, make_classification, make_regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

%matplotlib inline

np.set_printoptions(suppress=True, precision=4)
torch.manual_seed(123)
np.random.seed(123)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

print("device:", device)
print("PyTorch:", torch.__version__)

코드 설명:

- `np`: NumPy, 수치 계산에 사용합니다.
- `plt`: Matplotlib, 그래프 출력에 사용합니다.
- `torch`: PyTorch 기본 라이브러리입니다.
- `nn`: Neural Network 모듈입니다.
- `optim`: Optimizer를 사용할 때 필요합니다.
- `train_test_split`: 데이터를 학습용/테스트용으로 나눕니다.
- `device`: GPU가 있으면 GPU, 없으면 CPU를 사용합니다.

## 2. 회귀와 분류의 차이

강의에서는 회귀와 분류를 다음처럼 구분했습니다.

| 문제 유형 | 출력 | 예시 |
|---|---|---|
| 회귀 Regression | 연속적인 숫자 | 집값, 온도, 판매량 |
| 분류 Classification | 정해진 클래스 | 스팸/정상, 고양이/강아지 |

핵심 차이:

```text
회귀: 얼마나 많이?
분류: 무엇인가?
```

In [ ]:
problem_types = {
    "Regression": "연속적인 숫자를 예측하는 문제",
    "Classification": "정해진 클래스 중 하나를 예측하는 문제"
}

for key, value in problem_types.items():
    print(f"{key}: {value}")

이번 차시의 중심은 **회귀**입니다.

즉, 입력값 `x`를 받아 연속적인 숫자 `y`를 예측하는 문제입니다.

## 3. 선형 함수의 기본 형태

선형회귀의 기본 수식은 다음과 같습니다.

```text
y = Wx + b
```

- `x`: 입력값
- `W`: weight, 가중치
- `b`: bias, 편향
- `y`: 출력값

PyTorch에서는 이 구조를 `nn.Linear`로 구현합니다.

In [ ]:
def linear_formula(x, w, b):
    return w * x + b

x_sample = torch.tensor([1.0, 2.0, 3.0])
w_sample = torch.tensor(2.0)
b_sample = torch.tensor(1.0)

y_sample = linear_formula(x_sample, w_sample, b_sample)

print(y_sample)

코드 흐름:

```text
y = 2x + 1
x = [1, 2, 3]
y = [3, 5, 7]
```

선형 함수는 딥러닝에서 가장 기본이 되는 블록입니다.

## 4. `nn.Linear` 인스턴스 생성

`nn.Linear(in_features, out_features)`는 입력 차원을 출력 차원으로 바꾸는 선형 함수입니다.

In [ ]:
l = nn.Linear(2, 3)

print(l)

코드 설명:

- `in_features=2`: 입력 feature가 2개입니다.
- `out_features=3`: 출력 feature가 3개입니다.
- 내부적으로 weight와 bias를 가집니다.

수식 관점:

```text
y = xWᵀ + b
```

## 5. 1입력 1출력 선형 함수 만들기

가장 기본적인 `y = ax + b` 형태의 회귀 모델입니다.

In [ ]:
torch.manual_seed(123)

l1 = nn.Linear(1, 1)

print("l1 정보:")
print(l1)

print("\nl1의 파라미터:")
for name, param in l1.named_parameters():
    print("name:", name)
    print("tensor:", param)
    print("shape:", param.shape)

코드 설명:

- `nn.Linear(1, 1)`: 입력 1개를 받아 출력 1개를 만듭니다.
- `named_parameters()`: 모델 내부의 weight와 bias 이름과 값을 확인합니다.
- `requires_grad=True`: 학습 대상이라는 뜻입니다.

## 6. weight와 bias 직접 초기화

초깃값을 직접 설정하면 원하는 선형 함수를 만들 수 있습니다.

여기서는 다음 함수를 만듭니다.

```text
y = 2x + 1
```

In [ ]:
nn.init.constant_(l1.weight, 2.0)
nn.init.constant_(l1.bias, 1.0)

print("weight:", l1.weight)
print("bias:", l1.bias)

코드 설명:

- `nn.init.constant_()`: Tensor 값을 특정 상수로 채웁니다.
- `_`가 붙은 함수는 원본 Tensor를 직접 수정하는 in-place 함수입니다.
- weight를 2, bias를 1로 설정했으므로 `y = 2x + 1`이 됩니다.

## 7. 입력 데이터 준비와 shape 변경

`nn.Linear`는 보통 2차원 입력을 기대합니다.

```text
[데이터 개수, feature 수]
```

1개 feature를 가진 5개 데이터라면 shape은 `[5, 1]`이어야 합니다.

In [ ]:
x_np = np.arange(-2.0, 2.1, 1.0)

x = torch.tensor(x_np).float()
x = x.view(-1, 1)

print("x_np:", x_np)
print("x:")
print(x)
print("x shape:", x.shape)

코드 설명:

- `np.arange(-2.0, 2.1, 1.0)`: -2부터 2까지 1 간격으로 생성합니다.
- `torch.tensor(...).float()`: PyTorch Tensor로 변환합니다.
- `view(-1, 1)`: `[5]` 형태를 `[5, 1]`로 바꿉니다.
- `-1`: 나머지 차원은 PyTorch가 자동 계산합니다.

## 8. 1차 함수 테스트

앞에서 만든 `l1`에 입력 `x`를 넣어 `y = 2x + 1`이 계산되는지 확인합니다.

In [ ]:
y = l1(x)

print("y shape:", y.shape)
print("y data:")
print(y.data)

결과 해석:

입력:

```text
[-2, -1, 0, 1, 2]
```

공식:

```text
y = 2x + 1
```

출력:

```text
[-3, -1, 1, 3, 5]
```

## 9. 2입력 1출력 선형 함수

이번에는 입력이 2개이고 출력이 1개인 선형 함수를 만듭니다.

공식:

```text
y = x1 + x2 + 2
```

In [ ]:
l2 = nn.Linear(2, 1)

nn.init.constant_(l2.weight, 1.0)
nn.init.constant_(l2.bias, 2.0)

print("weight:", l2.weight)
print("bias:", l2.bias)

코드 설명:

- `nn.Linear(2, 1)`: 입력 2개를 받아 출력 1개를 만듭니다.
- weight가 모두 1, bias가 2이므로 `y = x1 + x2 + 2`입니다.

## 10. 2입력 1출력 테스트

4가지 입력 조합을 넣어 결과를 확인합니다.

In [ ]:
x2_np = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

x2 = torch.tensor(x2_np).float()

y2 = l2(x2)

print("x2 shape:", x2.shape)
print("x2:")
print(x2)

print("\ny2 shape:", y2.shape)
print("y2 data:")
print(y2.data)

결과 해석:

```text
[0, 0] → 0 + 0 + 2 = 2
[0, 1] → 0 + 1 + 2 = 3
[1, 0] → 1 + 0 + 2 = 3
[1, 1] → 1 + 1 + 2 = 4
```

## 11. 2입력 3출력 선형 함수

입력 2개로 출력 3개를 동시에 만들 수도 있습니다.

수식:

```text
y1 = 1*x1 + 1*x2 + 2
y2 = 2*x1 + 2*x2 + 2
y3 = 3*x1 + 3*x2 + 2
```

In [ ]:
l3 = nn.Linear(2, 3)

nn.init.constant_(l3.weight[0, :], 1.0)
nn.init.constant_(l3.weight[1, :], 2.0)
nn.init.constant_(l3.weight[2, :], 3.0)
nn.init.constant_(l3.bias, 2.0)

y3 = l3(x2)

print("weight:")
print(l3.weight)

print("\nbias:")
print(l3.bias)

print("\ny3 shape:", y3.shape)
print("y3 data:")
print(y3.data)

결과 해석:

입력 하나마다 출력이 3개씩 나옵니다.

즉, shape이 다음처럼 바뀝니다.

```text
입력 x2: [4, 2]
출력 y3: [4, 3]
```

이 구조는 여러 값을 동시에 예측하는 다중 출력 회귀에서도 사용됩니다.

## 12. 커스텀 모델 클래스 정의

PyTorch 모델은 보통 `nn.Module`을 상속해서 만듭니다.

핵심 메서드:

- `__init__`: 모델이 사용할 Layer를 선언
- `forward`: 데이터가 어떤 순서로 흐를지 정의

In [ ]:
class Net(nn.Module):
    def __init__(self, n_input, n_output):
        super().__init__()

        self.l1 = nn.Linear(n_input, n_output)

    def forward(self, x):
        x1 = self.l1(x)
        return x1

코드 설명:

- `class Net(nn.Module)`: PyTorch 모델 클래스를 정의합니다.
- `super().__init__()`: 부모 클래스 초기화입니다.
- `self.l1`: 선형 Layer입니다.
- `forward()`: 순전파 흐름입니다.

중요:

```text
net(inputs)를 호출하면 내부적으로 forward(inputs)가 실행된다.
```

## 13. 모델 인스턴스 생성과 예측

`Net` 클래스로 실제 모델 객체를 만들고 예측을 수행합니다.

In [ ]:
inputs_dummy = torch.ones(100, 1)

n_input = 1
n_output = 1

net = Net(n_input, n_output)

outputs_dummy = net(inputs_dummy)

print("outputs shape:", outputs_dummy.shape)
print(outputs_dummy[:5])

코드 설명:

- `inputs_dummy`: 100개 샘플, feature 1개
- `net = Net(1, 1)`: 1입력 1출력 모델
- `net(inputs_dummy)`: 모델을 함수처럼 호출합니다.
- 출력 shape은 `[100, 1]`입니다.

## 14. MSELoss와 경사 계산

회귀 문제에서는 MSELoss를 자주 사용합니다.

공식:

```text
Loss = mean((prediction - target)²)
```

In [ ]:
criterion = nn.MSELoss()

labels_dummy = torch.zeros(100, 1)

loss = criterion(outputs_dummy, labels_dummy)

loss.backward()

print("loss:", loss.item())
print("weight grad:", net.l1.weight.grad)
print("bias grad:", net.l1.bias.grad)

코드 설명:

- `criterion(outputs_dummy, labels_dummy)`: 예측값과 정답의 MSE를 계산합니다.
- `loss.backward()`: 역전파로 gradient를 계산합니다.
- `weight.grad`, `bias.grad`: 학습 방향 정보입니다.

## 15. 선형회귀용 데이터 생성

원본 강의에서는 보스턴 주택 가격 데이터의 RM, LSTAT 같은 feature를 사용했습니다.

여기서는 실행 안정성을 위해 scikit-learn의 `make_regression()`으로 비슷한 회귀 데이터를 생성합니다.

In [ ]:
X_reg, y_reg = make_regression(
    n_samples=506,
    n_features=1,
    noise=15.0,
    random_state=42
)

# 값 범위를 보기 쉽게 조정
X_reg = StandardScaler().fit_transform(X_reg)
y_reg = (y_reg - y_reg.mean()) / y_reg.std()

print("X_reg shape:", X_reg.shape)
print("y_reg shape:", y_reg.shape)
print("X sample:")
print(X_reg[:5])
print("y sample:")
print(y_reg[:5])

데이터 설명:

- `X_reg`: 입력 feature 1개
- `y_reg`: 예측해야 하는 연속값
- 원본 강의의 RM → 가격 예측 구조와 동일하게, 여기서는 1개 입력으로 1개 숫자를 예측합니다.

## 16. 회귀 데이터 산점도

입력값과 정답값의 관계를 확인합니다.

In [ ]:
plt.scatter(X_reg, y_reg, s=12)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Regression Data")
plt.show()

그래프 해석:

점들이 대략 직선 방향으로 분포하면 선형회귀 모델이 적합할 가능성이 있습니다.

선형회귀는 이 점들의 흐름을 가장 잘 대표하는 직선을 찾는 것이 목표입니다.

## 17. Tensor 변환과 shape 확인

PyTorch 모델 학습을 위해 NumPy 배열을 Tensor로 변환합니다.

In [ ]:
inputs = torch.tensor(X_reg).float().to(device)
labels = torch.tensor(y_reg).float().view(-1, 1).to(device)

print("inputs shape:", inputs.shape)
print("labels shape:", labels.shape)

코드 설명:

- `inputs`: `[N, 1]` 형태입니다.
- `labels`: `[N, 1]` 형태로 맞춥니다.
- `view(-1, 1)`: 손실 계산을 위해 예측값과 정답값의 shape을 맞춥니다.

## 18. 선형회귀 모델 정의와 초기화

입력 1개, 출력 1개의 선형회귀 모델을 만듭니다.

In [ ]:
class LinearRegressionNet(nn.Module):
    def __init__(self, n_input, n_output):
        super().__init__()
        self.l1 = nn.Linear(n_input, n_output)

        nn.init.constant_(self.l1.weight, 1.0)
        nn.init.constant_(self.l1.bias, 1.0)

    def forward(self, x):
        return self.l1(x)

n_input = inputs.shape[1]
n_output = 1

net = LinearRegressionNet(n_input, n_output).to(device)

print(net)

for name, param in net.named_parameters():
    print(name, param.data)

코드 설명:

- `n_input`: 입력 feature 개수입니다.
- `n_output`: 회귀값 1개입니다.
- weight와 bias를 모두 1로 초기화합니다.
- `named_parameters()`로 학습 대상 파라미터를 확인합니다.

## 19. 손실 함수와 Optimizer 설정

회귀 문제이므로 MSELoss를 사용합니다.

Optimizer는 SGD를 사용합니다.

In [ ]:
criterion = nn.MSELoss()

lr = 0.01

optimizer = optim.SGD(net.parameters(), lr=lr)

num_epochs = 1000

history = np.zeros((0, 2))

print("criterion:", criterion)
print("optimizer:", optimizer)

변수 설명:

- `criterion`: 손실 함수입니다.
- `lr`: learning rate, 학습률입니다.
- `optimizer`: 파라미터를 수정하는 도구입니다.
- `num_epochs`: 학습 반복 횟수입니다.
- `history`: epoch와 loss 기록용 배열입니다.

## 20. 학습 전 예측과 손실 확인

학습 전 모델이 얼마나 틀리는지 확인합니다.

In [ ]:
outputs = net(inputs)

loss = criterion(outputs, labels)

print("학습 전 loss:", loss.item())

학습 전에는 weight와 bias가 임의 또는 초기값이므로 예측이 좋지 않을 수 있습니다.

이 loss를 줄이는 것이 학습의 목표입니다.

## 21. 경사 계산과 파라미터 수정 1회 확인

학습 루프 전체 전에 한 번만 직접 실행해봅니다.

In [ ]:
optimizer.zero_grad()

outputs = net(inputs)
loss = criterion(outputs, labels)

loss.backward()

print("weight grad:", net.l1.weight.grad)
print("bias grad:", net.l1.bias.grad)

optimizer.step()
optimizer.zero_grad()

print("수정 후 weight:", net.l1.weight.data)
print("수정 후 bias:", net.l1.bias.data)

코드 흐름:

1. `optimizer.zero_grad()`: 이전 gradient 초기화
2. `outputs = net(inputs)`: 예측 계산
3. `loss = criterion(...)`: 손실 계산
4. `loss.backward()`: gradient 계산
5. `optimizer.step()`: 파라미터 수정
6. `optimizer.zero_grad()`: gradient 초기화

이 구조가 경사하강법의 핵심입니다.

## 22. 전체 학습 루프 구현

이제 1000번 반복하여 학습합니다.

In [ ]:
net = LinearRegressionNet(n_input, n_output).to(device)
criterion = nn.MSELoss()
optimizer = optim.SGD(net.parameters(), lr=0.01)

num_epochs = 1000
history = np.zeros((0, 2))

for epoch in range(num_epochs):
    optimizer.zero_grad()

    outputs = net(inputs)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        history = np.vstack((history, np.array([epoch, loss.item()])))

print("초기 손실값:", history[0, 1])
print("최종 손실값:", history[-1, 1])

for name, param in net.named_parameters():
    print(name, param.data)

학습 결과 해석:

- 초기 손실보다 최종 손실이 작아지면 학습이 진행된 것입니다.
- weight와 bias가 데이터를 잘 설명하는 방향으로 수정됩니다.

## 23. 학습 곡선 출력

손실값이 어떻게 줄어드는지 확인합니다.

In [ ]:
plt.plot(history[:, 0], history[:, 1])
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Training Loss")
plt.show()

그래프 해석:

- 우하향하면 학습이 잘 진행되는 것입니다.
- 손실이 발산하거나 `nan`이 나오면 학습률이 너무 클 수 있습니다.

## 24. 회귀 직선 출력

학습된 모델이 만든 직선을 데이터 위에 그립니다.

In [ ]:
x_line = np.array([X_reg.min(), X_reg.max()]).reshape(-1, 1)
x_line_t = torch.tensor(x_line).float().to(device)

with torch.no_grad():
    y_line_t = net(x_line_t)

plt.scatter(X_reg, y_reg, s=12)
plt.plot(x_line, y_line_t.cpu().numpy())
plt.xlabel("x")
plt.ylabel("y")
plt.title("Regression Line")
plt.show()

그래프 해석:

직선이 점들의 전체 흐름을 잘 따라가면 모델이 데이터를 잘 근사한 것입니다.

선형회귀의 목표는 산점도 위에서 가장 적절한 직선을 찾는 것입니다.

## 25. R² Score 직접 구현

R² Score는 모델이 데이터의 분산을 얼마나 설명하는지 나타내는 결정계수입니다.

공식:

```text
R² = 1 - SS_res / SS_tot
```

- `SS_res`: 잔차 제곱합
- `SS_tot`: 전체 변동량

In [ ]:
def r2_score_torch(y_true, y_pred):
    y_mean = y_true.mean()
    ss_tot = ((y_true - y_mean) ** 2).sum()
    ss_res = ((y_true - y_pred) ** 2).sum()
    return 1 - (ss_res / ss_tot)

with torch.no_grad():
    pred_all = net(inputs)
    r2 = r2_score_torch(labels, pred_all)

print("R² Score:", r2.item())

R² 해석:

- `R² ≈ 1`: 모델이 데이터를 매우 잘 설명합니다.
- `R² ≈ 0`: 평균만 예측하는 모델과 비슷합니다.
- `R² < 0`: 평균보다 못한 예측입니다.

주의:

```text
높은 R²가 인과관계를 의미하는 것은 아니다.
```

## 26. 2입력 다중 회귀로 확장

이번에는 입력 feature를 2개로 늘립니다.

원본 강의에서는 RM과 LSTAT 두 feature를 사용해 성능을 높였습니다.

여기서는 synthetic data에서 feature 2개를 사용합니다.

In [ ]:
X_multi, y_multi = make_regression(
    n_samples=506,
    n_features=2,
    noise=15.0,
    random_state=7
)

X_multi = StandardScaler().fit_transform(X_multi)
y_multi = (y_multi - y_multi.mean()) / y_multi.std()

inputs_multi = torch.tensor(X_multi).float().to(device)
labels_multi = torch.tensor(y_multi).float().view(-1, 1).to(device)

print("inputs_multi shape:", inputs_multi.shape)
print("labels_multi shape:", labels_multi.shape)

2입력 모델에서는 weight가 2개, bias가 1개입니다.

즉, 학습할 파라미터 수가 늘어납니다.

```text
y = w1*x1 + w2*x2 + b
```

## 27. 2입력 모델 학습

학습률 0.01로 학습해봅니다.

In [ ]:
net_multi = LinearRegressionNet(2, 1).to(device)
criterion = nn.MSELoss()
optimizer = optim.SGD(net_multi.parameters(), lr=0.01)

history_multi = np.zeros((0, 2))

for epoch in range(1000):
    optimizer.zero_grad()

    outputs = net_multi(inputs_multi)
    loss = criterion(outputs, labels_multi)

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        history_multi = np.vstack((history_multi, np.array([epoch, loss.item()])))

print("초기 손실값:", history_multi[0, 1])
print("최종 손실값:", history_multi[-1, 1])

결과 해석:

입력 feature가 늘어나면 모델이 더 많은 정보를 사용할 수 있습니다.

하지만 feature 값의 스케일이 크거나 학습률이 너무 크면 발산할 수 있습니다.

## 28. 학습률이 너무 클 때 발산하는 예시

학습률이 너무 크면 loss가 `inf` 또는 `nan`으로 발산할 수 있습니다.

In [ ]:
net_bad_lr = LinearRegressionNet(2, 1).to(device)
optimizer_bad = optim.SGD(net_bad_lr.parameters(), lr=5.0)

bad_losses = []

for epoch in range(20):
    optimizer_bad.zero_grad()

    outputs = net_bad_lr(inputs_multi)
    loss = criterion(outputs, labels_multi)

    loss.backward()
    optimizer_bad.step()

    bad_losses.append(loss.item())

print("큰 학습률 loss 기록:")
print(bad_losses)

해석:

- 학습률이 너무 크면 정답 지점을 지나쳐 버립니다.
- 반복할수록 loss가 커지거나 `nan`이 될 수 있습니다.
- 학습률은 매우 중요한 하이퍼파라미터입니다.

## 29. 학습률을 줄여 안정적으로 학습하기

학습률을 줄이면 안정적으로 수렴할 수 있습니다.

In [ ]:
net_good_lr = LinearRegressionNet(2, 1).to(device)
optimizer_good = optim.SGD(net_good_lr.parameters(), lr=0.001)

good_history = np.zeros((0, 2))

for epoch in range(1000):
    optimizer_good.zero_grad()

    outputs = net_good_lr(inputs_multi)
    loss = criterion(outputs, labels_multi)

    loss.backward()
    optimizer_good.step()

    if epoch % 50 == 0:
        good_history = np.vstack((good_history, np.array([epoch, loss.item()])))

print("초기 손실값:", good_history[0, 1])
print("최종 손실값:", good_history[-1, 1])

정리:

```text
학습률이 너무 큼 → 발산
학습률이 너무 작음 → 학습이 느림
적절한 학습률 → 안정적 수렴
```

## 30. Scikit-Learn 기본 분류 예제

강의 자료의 이론 부분에서는 Scikit-Learn과 PyTorch의 차이도 설명했습니다.

Scikit-Learn은 간단한 머신러닝 모델을 빠르게 구현하기 좋습니다.

In [ ]:
iris = load_iris()

X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

sk_model = LogisticRegression(max_iter=200)
sk_model.fit(X_train, y_train)

predictions = sk_model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print(f"Scikit-Learn Iris Accuracy: {accuracy * 100:.2f}%")

코드 흐름:

1. `load_iris()`: 데이터 불러오기
2. `train_test_split()`: 학습/테스트 데이터 분리
3. `LogisticRegression()`: 모델 생성
4. `fit()`: 학습
5. `predict()`: 예측
6. `accuracy_score()`: 정확도 평가

Scikit-Learn은 간단하고 직관적인 머신러닝 실습에 적합합니다.

## 31. PyTorch 이진 분류 예제

PyTorch는 모델 구조와 학습 루프를 직접 정의합니다.

복잡한 신경망이나 GPU 확장이 필요한 경우 유리합니다.

In [ ]:
X_bin, y_bin = make_classification(
    n_samples=1000,
    n_features=10,
    n_classes=2,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X_bin,
    y_bin,
    test_size=0.2,
    random_state=42
)

x_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

x_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

print(x_train_tensor.shape)
print(y_train_tensor.shape)

정답 `y`는 0 또는 1입니다.

이진 분류 모델은 보통 출력 1개를 만들고, sigmoid를 통해 0~1 사이 확률로 바꿉니다.

## 32. PyTorch 이진 분류 모델 정의

간단한 선형 이진 분류 모델입니다.

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(10, 1)

    def forward(self, x):
        return torch.sigmoid(self.layer(x))

binary_model = SimpleNN()

criterion_bce = nn.BCELoss()
optimizer_bce = optim.SGD(binary_model.parameters(), lr=0.01)

print(binary_model)

코드 설명:

- `nn.Linear(10, 1)`: 입력 feature 10개를 출력 1개로 바꿉니다.
- `torch.sigmoid()`: 출력값을 0~1 사이 확률로 바꿉니다.
- `BCELoss`: 이진 분류 손실 함수입니다.

## 33. PyTorch 이진 분류 학습

직접 학습 루프를 작성합니다.

In [ ]:
epochs = 300

for epoch in range(epochs):
    binary_model.train()

    optimizer_bce.zero_grad()

    y_pred = binary_model(x_train_tensor)
    loss = criterion_bce(y_pred, y_train_tensor)

    loss.backward()
    optimizer_bce.step()

    if epoch % 100 == 0:
        print(f"{epoch} | loss: {loss.item():.4f}")

PyTorch 학습 루프는 Scikit-Learn보다 길지만, 내부 과정을 직접 제어할 수 있습니다.

핵심 흐름:

```text
예측 → 손실 계산 → 역전파 → 파라미터 업데이트
```

## 34. PyTorch 이진 분류 평가

평가할 때는 gradient 계산이 필요 없으므로 `torch.no_grad()`를 사용합니다.

In [ ]:
binary_model.eval()

with torch.no_grad():
    y_test_pred = binary_model(x_test_tensor)
    y_test_pred = (y_test_pred > 0.5).float()

    accuracy = (y_test_pred.eq(y_test_tensor).sum().item()) / y_test_tensor.shape[0]

print(f"PyTorch Binary Test Accuracy: {accuracy * 100:.2f}%")

코드 설명:

- `model.eval()`: 평가 모드로 전환합니다.
- `torch.no_grad()`: gradient 계산을 끕니다.
- `> 0.5`: 확률이 0.5보다 크면 class 1, 아니면 class 0입니다.
- `eq()`: 예측과 정답이 같은지 비교합니다.

## 35. 과적합 Overfitting 개념

과적합은 모델이 훈련 데이터는 잘 맞추지만 새로운 데이터에는 약한 상태입니다.

징후:

```text
Train Loss는 계속 감소
Validation Loss는 어느 순간 증가
```

해결 방법:

- Ridge, Lasso 같은 정규화
- 데이터 추가
- 조기 종료 Early Stopping
- Dropout

In [ ]:
overfitting_summary = {
    "Overfitting": "훈련 데이터를 과도하게 암기해 새로운 데이터 성능이 떨어짐",
    "Symptom": "Train Loss는 감소하지만 Validation Loss는 증가",
    "Solution": "Regularization, Data Augmentation, Early Stopping"
}

for key, value in overfitting_summary.items():
    print(f"{key}: {value}")

## 36. Ridge Regression: L2 정규화

Ridge는 weight의 제곱합을 패널티로 추가합니다.

공식:

```text
Loss = MSE + λ||W||²
```

PyTorch에서는 Optimizer의 `weight_decay`로 L2 정규화를 적용할 수 있습니다.

In [ ]:
ridge_model = nn.Linear(2, 1)

lambda_l2 = 1e-2

ridge_optimizer = optim.SGD(
    ridge_model.parameters(),
    lr=0.1,
    weight_decay=lambda_l2
)

print(ridge_optimizer)

Ridge 특징:

- 모든 feature를 조금씩 사용합니다.
- weight를 0에 가깝게 줄이지만 완전히 0으로 만들지는 않습니다.
- 다중공선성이 있을 때 안정적인 예측에 도움이 됩니다.

## 37. Lasso Regression: L1 정규화

Lasso는 weight 절댓값 합을 패널티로 추가합니다.

공식:

```text
Loss = MSE + λ∑|W|
```

PyTorch Optimizer는 L1을 직접 지원하지 않기 때문에 loss에 직접 더해야 합니다.

In [ ]:
lasso_model = nn.Linear(2, 1)

sample_x = torch.randn(10, 2)
sample_y = torch.randn(10, 1)

mse_fn = nn.MSELoss()
lambda_l1 = 0.01

pred = lasso_model(sample_x)
mse_loss = mse_fn(pred, sample_y)

l1_norm = sum(
    param.abs().sum()
    for name, param in lasso_model.named_parameters()
    if "weight" in name
)

lasso_loss = mse_loss + lambda_l1 * l1_norm

print("MSE loss:", mse_loss.item())
print("L1 norm:", l1_norm.item())
print("Lasso loss:", lasso_loss.item())

Lasso 특징:

- 덜 중요한 feature의 weight를 0으로 만들 수 있습니다.
- 자동 feature selection 효과가 있습니다.
- 해석력이 중요한 고차원 데이터에 유용합니다.

## 38. ElasticNet: L1 + L2 결합

ElasticNet은 Ridge와 Lasso의 장점을 합친 방식입니다.

공식:

```text
Loss = MSE + α|W|₁ + λ|W|²
```

In [ ]:
elastic_model = nn.Linear(2, 1)

elastic_optimizer = optim.SGD(
    elastic_model.parameters(),
    lr=0.1,
    weight_decay=1e-2
)

pred = elastic_model(sample_x)
mse_loss = mse_fn(pred, sample_y)

l1_norm = sum(param.abs().sum() for param in elastic_model.parameters())

elastic_loss = mse_loss + 1e-3 * l1_norm

print("ElasticNet-style loss:", elastic_loss.item())

ElasticNet 특징:

- Lasso의 feature selection 효과
- Ridge의 안정성
- feature가 많고 서로 상관관계가 높은 데이터에서 유용합니다.

## 39. 주요 함수 / 변수 / 약어 정리

| 이름 | 의미 | 설명 |
|---|---|---|
| `Regression` | 회귀 | 연속적인 숫자를 예측 |
| `Classification` | 분류 | 정해진 클래스 중 하나를 예측 |
| `nn.Linear` | 선형 함수 | `y = xWᵀ + b` 계산 |
| `in_features` | 입력 특성 수 | 입력 feature 개수 |
| `out_features` | 출력 특성 수 | 출력 feature 개수 |
| `weight` | 가중치 | 입력의 영향력 |
| `bias` | 편향 | 기본 보정값 |
| `nn.Module` | 모델 부모 클래스 | PyTorch 모델의 기본 |
| `__init__` | 초기화 메서드 | Layer 선언 |
| `forward` | 순전파 메서드 | 데이터 흐름 정의 |
| `MSELoss` | 평균 제곱 오차 | 회귀 손실 함수 |
| `optimizer` | 최적화 함수 | 파라미터 수정 도구 |
| `SGD` | 경사하강법 계열 | Stochastic Gradient Descent |
| `lr` | 학습률 | 한 번에 이동하는 크기 |
| `R²` | 결정계수 | 모델 설명력 평가 |
| `Ridge` | L2 정규화 | weight 제곱합 패널티 |
| `Lasso` | L1 정규화 | weight 절댓값 패널티 |
| `ElasticNet` | L1+L2 | Lasso와 Ridge 결합 |

## 40. 시험용 요약

```text
선형회귀 = y = Wx + b
```

핵심 정리:

- 회귀는 연속적인 숫자를 예측하는 문제입니다.
- 분류는 정해진 클래스 중 하나를 예측하는 문제입니다.
- `nn.Linear(in_features, out_features)`는 선형 변환을 수행합니다.
- `nn.Linear`의 내부에는 weight와 bias가 있습니다.
- weight shape은 `[out_features, in_features]`입니다.
- bias shape은 `[out_features]`입니다.
- 1입력 1출력은 `y = wx + b`입니다.
- 2입력 1출력은 `y = w1x1 + w2x2 + b`입니다.
- `nn.Module`을 상속해서 커스텀 모델을 만듭니다.
- `__init__`에서는 Layer를 선언합니다.
- `forward()`에서는 데이터 흐름을 정의합니다.
- `net(inputs)`는 내부적으로 `forward(inputs)`를 실행합니다.
- 회귀 문제의 대표 손실 함수는 `MSELoss`입니다.
- 학습 루프는 예측 → 손실 계산 → 역전파 → 업데이트입니다.
- `optimizer.zero_grad()`는 gradient 초기화입니다.
- `loss.backward()`는 gradient 계산입니다.
- `optimizer.step()`은 파라미터 수정입니다.
- 학습률이 너무 크면 loss가 발산할 수 있습니다.
- R² Score는 모델이 데이터 변동성을 얼마나 설명하는지 나타냅니다.
- Ridge는 L2 정규화, Lasso는 L1 정규화입니다.
- ElasticNet은 L1과 L2를 결합한 정규화입니다.